<a href="https://colab.research.google.com/github/ramkiran21/Codesoft/blob/IMDB-MOVIE-RATING/MOVIE_RATING_PREDICTION_WITH_PYTHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Rating Prediction Project

In [12]:
import pandas as pd

# Load the dataset with 'latin1' encoding
try:
    df = pd.read_csv('/content/IMDb Movies India.csv', encoding='latin1')
    print("Dataset loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print("Error: '/content/IMDb Movies India.csv' not found. Please ensure the file is in the correct path.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

Dataset loaded successfully!


,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


## Data Preprocessing and Exploration

In [13]:
print("Dataset Info:")
df.info()

print("\nMissing Values:")
df.isnull().sum()


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  object 
 2   Duration  7240 non-null   object 
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   object 
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(1), object(9)
memory usage: 1.2+ MB

Missing Values:


,0
Name,0
Year,528
Duration,8269
Genre,1877
Rating,7590
Votes,7589
Director,525
Actor 1,1617
Actor 2,2384
Actor 3,3144


## Data Cleaning and Type Conversion

In [14]:
# Clean and convert 'Year' column
df['Year'] = df['Year'].astype(str).str.extract('(\d{4})').astype(float)

# Clean and convert 'Duration' column
df['Duration'] = df['Duration'].astype(str).str.replace(' min', '').astype(float)

# Clean and convert 'Votes' column, handling potential non-numeric characters like '$' and 'M'
df['Votes'] = df['Votes'].astype(str).str.replace(',', '').str.replace('$', '', regex=False).str.replace('M', '', regex=False)
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce') # Convert to numeric, coercing errors to NaN

print("Data types after initial cleaning:")
df.info()

Data types after initial cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15509 entries, 0 to 15508
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Name      15509 non-null  object 
 1   Year      14981 non-null  float64
 2   Duration  7240 non-null   float64
 3   Genre     13632 non-null  object 
 4   Rating    7919 non-null   float64
 5   Votes     7920 non-null   float64
 6   Director  14984 non-null  object 
 7   Actor 1   13892 non-null  object 
 8   Actor 2   13125 non-null  object 
 9   Actor 3   12365 non-null  object 
dtypes: float64(4), object(6)
memory usage: 1.2+ MB


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_14071/572431459.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['Year'] = df['Year'].astype(str).str.extract('(\d{4})').astype(float)


### Handling Missing Values
We will remove rows missing the target variable ('Rating') and impute missing values for other features.

In [15]:
# Drop rows where Rating is NaN
df = df.dropna(subset=['Rating'])

# Impute numerical columns with median
df['Duration'] = df['Duration'].fillna(df['Duration'].median())
df['Votes'] = df['Votes'].fillna(df['Votes'].median())
df['Year'] = df['Year'].fillna(df['Year'].median())

# Impute categorical columns with 'Unknown'
cat_cols = ['Genre', 'Director', 'Actor 1', 'Actor 2', 'Actor 3']
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')

print("Missing values after imputation:")
print(df.isnull().sum())
display(df.head())

Missing values after imputation:
Name        0
Year        0
Duration    0
Genre       0
Rating      0
Votes       0
Director    0
Actor 1     0
Actor 2     0
Actor 3     0
dtype: int64


,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
1,#Gadhvi (He thought he was Gandhi),2019.0,109.0,Drama,7.0,8.0,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
3,#Yaaram,2019.0,110.0,"Comedy, Romance",4.4,35.0,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
5,...Aur Pyaar Ho Gaya,1997.0,147.0,"Comedy, Drama, Musical",4.7,827.0,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor
6,...Yahaan,2005.0,142.0,"Drama, Romance, War",7.4,1086.0,Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma
8,?: A Question Mark,2012.0,82.0,"Horror, Mystery, Thriller",5.6,326.0,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia


## Feature Engineering
We will encode categorical features using the mean of the target variable ('Rating') to transform them into numerical values.

In [16]:
# Genre: Calculate the mean rating for each genre
genre_mean_rating = df.groupby('Genre')['Rating'].transform('mean')
df['Genre_Encoded'] = genre_mean_rating

# Director: Calculate the mean rating for each director
director_mean_rating = df.groupby('Director')['Rating'].transform('mean')
df['Director_Encoded'] = director_mean_rating

# Actor 1: Calculate the mean rating for each primary actor
actor1_mean_rating = df.groupby('Actor 1')['Rating'].transform('mean')
df['Actor1_Encoded'] = actor1_mean_rating

# Actor 2: Calculate the mean rating for each secondary actor
actor2_mean_rating = df.groupby('Actor 2')['Rating'].transform('mean')
df['Actor2_Encoded'] = actor2_mean_rating

# Actor 3: Calculate the mean rating for each tertiary actor
actor3_mean_rating = df.groupby('Actor 3')['Rating'].transform('mean')
df['Actor3_Encoded'] = actor3_mean_rating

print("Features encoded successfully. Preview of new columns:")
display(df[['Rating', 'Genre_Encoded', 'Director_Encoded', 'Actor1_Encoded']].head())

Features encoded successfully. Preview of new columns:


,Rating,Genre_Encoded,Director_Encoded,Actor1_Encoded
1,7.0,6.352082,7.000000,6.850000
3,4.4,5.722500,4.400000,5.420000
5,4.7,6.224490,5.358824,4.788889
6,7.4,6.820000,7.500000,5.356000
8,5.6,5.463636,5.600000,5.600000


## Model Building and Evaluation
We will use a Random Forest Regressor to predict movie ratings based on our engineered features.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Selecting the features for the model
features = ['Year', 'Duration', 'Votes', 'Genre_Encoded', 'Director_Encoded', 'Actor1_Encoded', 'Actor2_Encoded', 'Actor3_Encoded']
X = df[features]
y = df['Rating']

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initializing and training the Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Making predictions
y_pred = model.predict(X_test)

# Evaluating the model
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")
print(f"R-squared Score: {r2:.4f}")

Mean Squared Error: 0.3754
Mean Absolute Error: 0.4097
R-squared Score: 0.7981


#The movie rating prediction model is now complete!

We achieved an R-squared score of 0.7981, which means our model explains roughly 80% of the variance in movie ratings. The Mean Absolute Error (MAE) is 0.4097, indicating that on average, our predictions are off by only about 0.4 points on the 1-10 rating scale.

This suggests that factors like the director's track record, the main actors, and the genre are indeed strong predictors of how a movie will be rated. Would you like to try making a prediction for a specific movie, or perhaps visualize which features were the most important for the model?